In [1]:
import ast
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sqlite3
import json
pd.set_option('display.max_columns', None)

# convert r-string to pandas compatible
def r_vector_to_list(x):
    if isinstance(x, str) and x.startswith("c("):
        x = x.replace("c(", "[").replace(")", "]")

        # deal with NA values in RecipeIngredientQuantities
        x = re.sub(r'\bNA\b', 'None', x)

    # will return only a value so use df.map()!
        return ast.literal_eval(x)
    return x

In [13]:
df_recipes = pd.read_csv('./data/raw/recipes.csv')
df_reviews = pd.read_csv('./data/raw/reviews.csv')

In [4]:
print("Total number of rows:", len(df_recipes))
print("Total number of unique recipe IDs:", df_recipes["RecipeId"].nunique())

Total number of rows: 522517
Total number of unique recipe IDs: 522517


In [5]:
# convert R string to pandas compatible format
c_column = ["Keywords", "Images", "RecipeInstructions","RecipeIngredientQuantities", "RecipeIngredientParts",]
df_recipes[c_column] = df_recipes[c_column].map(r_vector_to_list)
# convert to JSON for SQLite storage
for col in c_column:
    df_recipes[col] = df_recipes[col].apply(
        lambda x: json.dumps(x) if isinstance(x, list) else x
    )

# standardize unphysical values for nutrition to NaN
cols = [
    "Calories", "FatContent", "SaturatedFatContent", "CholesterolContent",
    "SodiumContent", "CarbohydrateContent", "FiberContent",
    "SugarContent", "ProteinContent"
]
df_recipes[cols] = df_recipes[cols].mask(df_recipes[cols] <= 0)

# same for recipe serving
df_recipes["RecipeServings"] = df_recipes["RecipeServings"].mask(df_recipes["RecipeServings"] <= 0)

In [6]:
# calculate nutrition per serving
cols = [
    "Calories", "FatContent", "SaturatedFatContent", "CholesterolContent",
    "SodiumContent", "CarbohydrateContent", "FiberContent",
    "SugarContent", "ProteinContent"
]

for col in cols:
    new_col = f"{col}PerServing"
    
    # calculate per serving
    values = df_recipes[col] / df_recipes["RecipeServings"]
    
    pos = df_recipes.columns.get_loc(col) + 1 # place it next to the respective metric
    df_recipes.insert(pos, new_col, values)

In [7]:
# convert ISO 8601 to pandas compatible
time_cols = ["CookTime", "PrepTime", "TotalTime"]
for col in time_cols:
    # errors argument is used since there are a couple of NaNs
    df_recipes[col] = pd.to_timedelta(df_recipes[col], errors="coerce")
    df_recipes[col] = df_recipes[col].dt.total_seconds() # convert to seconds for easy operation
df_recipes["DatePublished"] = pd.to_datetime(df_recipes["DatePublished"], errors="coerce")

# same for df_reviews
review_date_cols = ["DateSubmitted", "DateModified"]
for col in review_date_cols:
    df_reviews[col] = pd.to_datetime(df_reviews[col], errors="coerce")

# add a column indicating whether the total time is consistent with cook_time + prep_time (just for sanity check)
diff = (df_recipes["CookTime"] + df_recipes["PrepTime"]) - df_recipes["TotalTime"]
consistent_time = np.where(
    diff <= 60., # less than a minute is consistent
    "consistent",
    np.where((diff > 60) & (diff < 900), "partially missing", "inconsistent") # between 1 and 15 minutes is partially missing
)
df_recipes.insert(7, "ConsistentTime", consistent_time) # insert into right of TotalTime

df_recipes["DatePublished"] = df_recipes["DatePublished"].astype(str) # convert to string for SQLite storage
df_reviews[review_date_cols] = df_reviews[review_date_cols].astype(str) # same

In [8]:
# binning the total time
bins = [
    0. * 60.,
    15 * 60., # 15 mins
    30 * 60., # 30 mins
    60 * 60., # 1 hr
    120 * 60., # 2 hrs
    240 * 60., # 4 hrs
    999999999 * 60., # max.
]

labels = [
    "<15 mins",
    "15-30 mins",
    "30-60 mins",
    "1-2 hrs",
    "2-4 hrs",
    ">4 hrs"
]

df_recipes.insert(8, 
                "TotalTimeBucket",
                pd.cut(
                    df_recipes["TotalTime"],
                    bins=bins,
                    labels=labels
                ),
)


In [9]:
# create tables for flag missing values
df_recipes_flag = df_recipes[[
    "TotalTime","RecipeCategory","DatePublished",
    "RecipeIngredientParts","AggregatedRating",
    "ReviewCount","RecipeServings","RecipeYield","RecipeInstructions","Calories",
    "FatContent", "SaturatedFatContent", "CholesterolContent", "SodiumContent", "CarbohydrateContent", "FiberContent", "SugarContent", "ProteinContent"
]].notna().add_prefix("Has")

df_recipes = pd.concat([df_recipes, df_recipes_flag], axis=1) # concatenate

In [11]:
conn = sqlite3.connect("food_recipe.db")
df_recipes.to_sql("recipes", conn, if_exists='replace', index=False)
df_reviews.to_sql("reviews", conn, if_exists='replace', index=False)
conn.close()